# The gradient of the balance loss

The lesson's `balance_loss` returns a number. A router trained with it needs that number's
gradient, first with respect to the router logits and then with respect to the router's weights.
You will derive both, check them against finite differences, and then train a collapsed router on
the balance loss alone.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np


def softmax(z):
    """Probabilities along the last axis, after subtracting each row's maximum."""
    shifted = z - np.max(z, axis=-1, keepdims=True)
    weights = np.exp(shifted)
    return weights / np.sum(weights, axis=-1, keepdims=True)


def top_k(logits, k):
    """Each token's k chosen experts: the indices of its k largest logits."""
    return np.argsort(-logits, axis=1)[:, :k]


def balance_loss(logits, k, alpha):
    """alpha * N * sum_i f_i P_i for a batch of router logits of shape (T, N)."""
    T, N = logits.shape
    f = np.bincount(top_k(logits, k).ravel(), minlength=N) / (k * T)
    P = softmax(logits).mean(axis=0)
    return alpha * N * float(f @ P)


def loads(logits, k):
    """Assignments per expert, as a multiple of the fair share k * T / N."""
    T, N = logits.shape
    return np.bincount(top_k(logits, k).ravel(), minlength=N) / (k * T / N)


# 512 tokens of width 8 from four clusters, all shifted the same way, and a router
# for six experts whose first row points along the shift, so expert 0 wins most tokens.
rng = np.random.default_rng(12)
T, d, N, k = 512, 8, 6, 2
centers = rng.normal(size=(4, d)) * 1.5
X = centers[rng.integers(4, size=T)] + rng.normal(size=(T, d)) * 0.7 + 1.0
Wg0 = rng.normal(size=(N, d)) * 0.1
Wg0[0] += 0.6
print("loads / fair share:  ", np.round(loads(X @ Wg0.T, k), 2))
print(f"balance loss / alpha: {balance_loss(X @ Wg0.T, k, 1.0):.3f}")

## Part 1 (coding task): The gradient with respect to the logits

Write `balance_grad(logits, k, alpha)`, the gradient of `balance_loss(logits, k, alpha)` with
respect to `logits`: an array of shape $(T, N)$ whose entry $(t, j)$ is $\partial \mathcal{L}_{\text{aux}} / \partial z_{tj}$. Hold the token fractions $f$ fixed, as training does. A small nudge of the logits changes no token's top $k$ unless two logits tie, so the finite differences in the check see the same $f$ as well.

Derive the formula on paper first, then write it without a loop over tokens.

In [ ]:
def balance_grad(logits, k, alpha):
    """d balance_loss / d logits, shape (T, N), with the fractions f held fixed."""
    T, N = logits.shape
    # TODO
    raise NotImplementedError

In [ ]:
def numerical_grad(fun, x, eps=1e-6):
    """Central differences of fun at x, one entry at a time."""
    grad = np.zeros_like(x)
    for i in np.ndindex(x.shape):
        step = np.zeros_like(x)
        step[i] = eps
        grad[i] = (fun(x + step) - fun(x - step)) / (2 * eps)
    return grad


Z = np.random.default_rng(1).normal(size=(9, 5))
G = np.asarray(balance_grad(Z, 2, 0.3), dtype=float)
assert G.shape == Z.shape, f"expected shape {Z.shape}, got {G.shape}"
G_num = numerical_grad(lambda z: balance_loss(z, 2, 0.3), Z)
assert np.allclose(G.sum(axis=1), 0, atol=1e-9), (
    "each row should add to 0: adding the same number to all of a token's logits leaves its "
    "probabilities, and so the loss, unchanged"
)
scale = float((G * G_num).sum() / (G_num * G_num).sum())
assert not np.isclose(scale, -1, atol=1e-3), "the sign is reversed"
assert not np.isclose(scale, 2, atol=1e-3), "twice too large with k = 2: the fractions f should add to 1 over the experts"
assert not np.isclose(scale, 9, atol=1e-3), "9 times too large on 9 tokens: P is a mean over the tokens"
assert np.allclose(G, G_num, atol=1e-8), "the gradient disagrees with finite differences of balance_loss"
print("Part 1 passed")

## Part 2 (coding task): Training the router on the balance loss alone

The router computes its logits as `X @ Wg.T`, one row of `Wg` per expert. Write

- `router_grad(X, Wg, k, alpha)`: the gradient of `balance_loss(X @ Wg.T, k, alpha)` with respect
  to `Wg`, of the same shape as `Wg`.
- `train_router(X, Wg, k, alpha, lr, steps)`: starting from a copy of `Wg`, take `steps` steps of
  gradient descent on the balance loss alone, $W_g \leftarrow W_g - \text{lr} \cdot \partial \mathcal{L}_{\text{aux}} / \partial W_g$, routing the tokens again at every step, and
  return the trained matrix.

The check trains the router from the setup for 200 steps with $\alpha = 1$ and a learning rate of 0.2. With no task loss in the run, $\alpha$ only scales the step.

In [ ]:
def router_grad(X, Wg, k, alpha):
    # TODO
    raise NotImplementedError


def train_router(X, Wg, k, alpha, lr, steps):
    # TODO
    raise NotImplementedError

In [ ]:
check_rng = np.random.default_rng(2)
Xs, Ws = check_rng.normal(size=(6, 3)), check_rng.normal(size=(4, 3))
Gw = np.asarray(router_grad(Xs, Ws, 2, 0.5), dtype=float)
assert Gw.shape == Ws.shape, f"expected the shape of Wg, {Ws.shape}, got {Gw.shape}"
Gw_num = numerical_grad(lambda w: balance_loss(Xs @ w.T, 2, 0.5), Ws)
assert np.allclose(Gw, Gw_num, atol=1e-8), "router_grad disagrees with finite differences of balance_loss(X @ Wg.T, ...)"

before = Wg0.copy()
Wg = train_router(X, Wg0, k, 1.0, 0.2, 200)
assert np.array_equal(Wg0, before), "train_router changed the Wg it was given: work on a copy"
start, end = loads(X @ Wg0.T, k), loads(X @ Wg.T, k)
print("loads / fair share before:", np.round(start, 2))
print("loads / fair share after: ", np.round(end, 2))
print(f"balance loss / alpha: {balance_loss(X @ Wg0.T, k, 1.0):.4f} -> {balance_loss(X @ Wg.T, k, 1.0):.4f}")
assert end.max() < start.max(), "the busiest expert got busier: a descent step moves against the gradient"
assert end.max() <= 1.1 and end.min() >= 0.9, "after 200 steps every expert should be within 10% of its fair share"
print("Part 2 passed")

## Solution

### Part 1: The gradient with respect to the logits

In [ ]:
def balance_grad(logits, k, alpha):
    """d balance_loss / d logits, shape (T, N), with the fractions f held fixed."""
    T, N = logits.shape
    f = np.bincount(top_k(logits, k).ravel(), minlength=N) / (k * T)
    g = softmax(logits)
    seen = g @ f                                   # each token's probability-weighted load
    return alpha * N / T * g * (f[None, :] - seen[:, None])

**Why it works.** With $f$ fixed the loss is linear in the probabilities, $\mathcal{L} = \frac{\alpha N}{T} \sum_t \sum_i f_i\, g_{ti}$, and the softmax derivative $\partial g_{ti} / \partial z_{tj} = g_{ti}(\delta_{ij} - g_{tj})$ gives

$$
\frac{\partial \mathcal{L}}{\partial z_{tj}} = \frac{\alpha N}{T}\, g_{tj} \Big( f_j - \sum_i f_i\, g_{ti} \Big).
$$

The sum is the load token $t$ sees, the experts' fractions weighted by its own probabilities. A descent step lowers $z_{tj}$ when expert $j$ is busier than that and raises it when expert $j$ is
quieter, so every token moves probability toward the experts that are less loaded than its
current mix. The factor $g_{tj}$ means a token barely moves the logits of experts it already
gives almost no probability. Each row adds to zero, as the check asked, because
$\sum_j g_{tj} f_j$ is exactly the term subtracted.

### Part 2: Training the router on the balance loss alone

In [ ]:
def router_grad(X, Wg, k, alpha):
    G = balance_grad(X @ Wg.T, k, alpha)           # (T, N)
    return G.T @ X                                 # (N, d), the shape of Wg


def train_router(X, Wg, k, alpha, lr, steps):
    W = Wg.copy()
    for _ in range(steps):
        W -= lr * router_grad(X, W, k, alpha)
    return W

**Why it works.** The logits are $Z = X W_g^\top$, so $z_{tj} = \mathbf{w}_j \cdot \mathbf{x}_t$ and $\partial \mathcal{L} / \partial \mathbf{w}_j = \sum_t (\partial \mathcal{L} / \partial z_{tj})\, \mathbf{x}_t$,
which is `G.T @ X` for the matrix `G` from Part 1. The routing has to be redone at every step,
because $f$ is what tells the gradient which experts are busy: `router_grad` recounts it each
time it is called.

Trained on nothing but the balance loss, the router goes from loads of
3.00, 0.17, 0.63, 1.96, 0.01 and 0.23 times the fair share, with expert 0 in every token's top two
and expert 4 in almost none, to within 3% of fair for all six experts, and the loss falls from
$2.75\alpha$ to about $\alpha$. In real training the task loss pulls the other way, and $\alpha$ sets how much
imbalance the router may trade for a lower task loss.